# PANOPLY Workbench Startup Notebook

This notebook walks you through uploading, validating, and configuring PANOPLY proteogenomic 
data on a **Manifold** workbench. Upon completion, the notebook will produce an **`inputs.json`** 
file that can be used to launch a PANOPLY workflow, with appropriate S3 file-paths and 
key parameters. By default, the `panoply_unified_workflow` is assumed default, but any workflow 
in the [PANOPLY GitHub repo](https://github.com/broadinstitute/PANOPLY) can be used. The notebook 
is designed with flexibility and repeated usage in mind; if you need to regenerate a JSON 
for a new sample subset, it is recommended to reuse this notebook.

-----
### Using this notebook
1. This notebook requires an **R kernel** (IRkernel). If you are **not** running through Manifold, and
   your environment only offers a Python kernel, install one first from a terminal, 
   e.g. `mamba install -c conda-forge r-irkernel`. Once installed, select the R kernel for this notebook.
2. Run the **Setup** cell once per session -- it installs any R packages this notebook needs
   that aren't already present. A couple are large Bioconductor packages used for gene/protein-ID
   conversion, so the very first run can take a while; subsequent runs are fast.
3. This notebook works in terms of **sessions** -- see the *Sessions* section right after Setup
   for what that means and how to resume, restart, or reload one.
4. Run cells top to bottom the first time. Several cells prompt for input in the console below
   the cell -- read the preceding text before running each one.

### Prepare your data
Place (or upload) the following into `~/workbench/inputs/`:
* At least **one proteomics dataset** (global proteome, phosphoproteome, acetylome, and/or
  ubiquitylome), in [GCT v1.3 format](https://clue.io/connectopedia/gct_format).
* An `Annotation` CSV with at least `Sample.ID` and `Type` columns, plus any other sample
  annotations you wish to analyze.
* Optionally:
  * Genomics data -- CNA and RNA, also GCT v1.3. Note that, for the unified pipeline, you should either 
  provide *both* or *neither*; you cannot currently analyze only one genomic datatype.
  * a `groups` CSV, used to subset the annotation file to annotations of interest. Should contain
  a single headerless column, with one annotation name per row. 
  * your own parameter YAML, if you have pre-edited the PANOPLY default `master-parameters.yaml`
  * PTM-SEA or GSEA `gmt` pathway databases (if you wish to use DBs other than the default)
  * reference proteome FASTA file, if you intend to run ClumpsPTM

These are your **originals** -- the notebook copies them into your session and only ever
modifies the copies, so you can always come back to exactly what you uploaded. Files can also be 
uploaded in a `.zip` file; the notebook will automatically detect any `.zip` in the `inputs/` folder.

Sample IDs (GCT column names) must match the `Sample.ID`s in your annotation table. See the
[PANOPLY wiki](https://github.com/broadinstitute/PANOPLY/wiki) for more on data formats.


### Setup
Run once per session. `wb_setup()` installs any R packages this notebook needs that aren't
already present (a couple are large Bioconductor packages used for gene/protein-ID conversion,
so the very first run can take a while; subsequent runs are fast), then loads the helper module.

In [ ]:
source("workbench-src/config.r")
wb_setup()

### Sessions
As you upload data and configure settings through this notebook, it is saved 
as a **session**: a complete, self-contained snapshots of one notebook run 
(stored in `~/workbench/workbench-setup/sessions/`). It is recommended to make 
a new session for each new dataset that you run. 

* **`current-session`** is the one you're actively working in. It's always safe to edit -- 
  nothing here is final until you explicitly name and save it (see *Finalized and
  Run Session* near the end). Your session current can always be reloaded, even if 
  you close the environment.
* Once you've named and saved a session (e.g. `odg-v4`), it can be used to generate an 
  `inputs.json` to run workflows. If you need to make edits to a saved workflow, 
  pick "load a saved session" below -- this restores it into `current-session` so you can
  keep working from exactly that point.

Run the cell below to choose: resume `current-session` (pick up where you left off), start a
new one (reset `current-session` and begin fresh), or load a previously saved one 
(overwrites `current-session`).

In [ ]:
state <- wb_load_state()

#### Just need to regenerate `inputs.json` for an existing saved session?
The cell above fully copies a saved session into `current-session/`, which is slow for large
GCTs -- and unnecessary if all you want is a fresh `inputs.json` for a saved session. 
Instead, run `state <- wb_open_saved_session()` below, then skip straight down to the *Generate
`inputs.json`* section near the end -- you don't need to re-run anything in between.

**WARNING:** This method should *only* be used to regenerate `inputs.json`! Don't try to map new inputs, 
create subsets, edit groups/colors, etc. -- those all write into `current-session/`, which is intentionally
left untouched. If you need to make larger edits to a saved session, use the cell above to 
reload it instead.

In [ ]:
# state <- wb_open_saved_session()

### Configuration
Two things you may want to change:
* `GITHUB_REF` -- the branch/tag of [broadinstitute/PANOPLY](https://github.com/broadinstitute/PANOPLY)
  that workflow WDLs and the default `master-parameters.yaml` are fetched from.
* `TARGET_WORKFLOW` -- which PANOPLY workflow to build `inputs.json` for. Defaults to
  `panoply_unified_workflow`. Run `wb_list_github_workflows()` to see other options.

In [ ]:
# wb_list_github_workflows()

In [ ]:
GITHUB_REF      <- "issue-githubWDL"
TARGET_WORKFLOW <- "panoply_unified_workflow"

state$github_ref      <- GITHUB_REF
state$target_workflow <- TARGET_WORKFLOW
state <- wb_save_state(state)

# Inputs
Run the cell below to map each file you've placed in `~/workbench/inputs/` to a data category
-- the available categories are listed for you as part of the prompt. A suggestion is offered
for filenames that appear to match, e.g. `...-proteome-....gct`; press Enter to accept
it or type a different number. Each category can hold only one file; mapping a second file to
the same category overwrites the first.

If you have a single ZIP file instead of individual files, just place it in `~/workbench/inputs/`
too -- it's detected automatically and you'll be asked whether to unzip and use it, then whether
to also map any other loose files in that folder.

In [ ]:
state <- wb_load_and_map_inputs(state)

# Validation
Checks the annotation table for required columns and unique `Sample.ID`s, checks sample-ID
overlap between the annotation table and every mapped GCT file, and checks (or interactively
helps you fix) the gene-ID column in each proteomics/genomics GCT file.

In [ ]:
state <- wb_validate_inputs(state)

# Data Processing
Toggles proteomics normalization/filtering, and (if the relevant data was mapped above) PTM-SEA
and MetaboAnalyst. If you opt into PTM-SEA or MetaboAnalyst, you'll be prompted to confirm or
select the relevant ID column.

In [ ]:
state <- wb_select_preprocessing_options(state)

# COSMO Label Selection (optional)
COSMO (COrrection of Sample Mislabeling by Omics) needs 1-3 clinical attributes that are binary,
well-balanced, and free of NAs. You'll be shown the valid candidates from your annotation table
and asked to pick from among them.

In [ ]:
state <- wb_select_cosmo_attributes(state)

# Clumps-PTM Setup (optional)
Only offered if at least 2 of {phosphoproteome, acetylome, ubiquitylome} were mapped above.
Requires a reference FASTA (matching the accession-ID type used in your PTM data). If you
mapped one to the `clumpsFASTA` category in the *Inputs* section above, it's used automatically
-- otherwise you'll be prompted for its local path below (e.g. `~/workbench/inputs/reference.fasta`
or `workbench/inputs/reference.fasta`; `s3://` paths aren't accepted here -- upload the file
locally and provide that path instead).

In [ ]:
state <- wb_select_clumpsptm_groups(state)

# Groups
Groups are the categorical annotations used for association and enrichment analysis. By default,
all valid annotation columns are used (or the columns listed in your `groups` file, if you
provided one) -- pass an explicit `columns = c(...)` to override. Annotations with more than
`max_categories` unique values are excluded (or treated as continuous, if numeric).

In [ ]:
wb_list_annotation_columns(state)

In [ ]:
# state <- wb_select_groups(state, columns = c("Type", "Stage"), max_categories = 10)
state <- wb_select_groups(state, max_categories = 10)

## Color Schemes (optional)
Modify the color-palette to be used in figures for your categorical annotations of interest. 
The default palette is pulled from the Paul Tol color palette, and is intended to be visually 
distinct and color-blind friendly. Default values can be updated in the `Edit a color` cell below.

### See the current color scheme

In [ ]:
wb_show_colors(state)

### Reset colors to defaults
Colors are assigned automatically based on the number of unique values per group; NA is always grey.

In [ ]:
state <- wb_reset_colors(state)

### Edit a color
Interactively pick a group, then either set one value's color at a time or replace every color
for that group at once from a comma-separated hex list. Repeats until you type 'quit'.

In [ ]:
state <- wb_edit_color(state)

# Sample Subsets
In order to streamline rerunning analysis on subsets of data, this notebook allows you to define 
sample subsets, based on samples' clinical annotation values. For each subset, GCT and 
CSV files will be filtered down to matching samples, and stored in a local folder under 
`current-session/subsets/<name>/`. Each time the `input.json` file is generated or edited, 
you will be prompted to choose a sample subset -- this allows you to easily swap sample-sets 
while keeping other parameters the same.

An `all` subset (every sample) is always created; this subset can't be removed or overwritten.
You can additionally:
* Add a new subset (picking an annotation column, the value(s) to include, 
and a name -- reusing an existing name overwrites it)
* Remove an existing subset
* Refresh all existing subsets using current session data (**note:** if any existing subset look out of date (its source data or group-column selection changed since it was last built), you'll be offered the chance to regenerate just those before the menu appears)

In [ ]:
state <- wb_create_subset(state)

# Finalize Parameters
Builds a customized `master-parameters.yaml`, merging your selected groups/colors/toggles with 
PANOPLY's default module parameters (either fetched from `GITHUB_REF`, or your own 
uploaded parameter file, if you provided one).

In [ ]:
master_params_path <- wb_build_master_parameters_yaml(state)
master_params_path

# Finalized and Run Session
The next cells will set up a finalized, named session (`sessions/<name>/`); this named 
session will have stable filepaths, and can be used to generate an input-json for
the selected WDL.

## Name and Save This Session
Name and save your session to create a stable snapshot of your uploaded files, configured 
parameters, and sample-subsets. Once a session has been saved, it can be used to generate an
`inputs.json`, which references files by their `s3://` path. After this point, a new session can 
safely be created for a new dataset, without clobbering existing filepaths.

If you have made changes to your `current-session` that you want to propogate to runtime, 
make sure to re-save your session before re-generating an `inputs.json`!

In [ ]:
state <- wb_save_session(state)

## Workflow Run Options
Any remaining toggles `TARGET_WORKFLOW` requires that aren't already inferred from
your data above (PTM-SEA, MetaboAnalyst, Clumps-PTM, normalization/filtering) -- you'll be
prompted for each one its WDL actually declares as required, so this adapts automatically if
`TARGET_WORKFLOW` is changed to something other than `panoply_unified_workflow`.

In [ ]:
state <- wb_select_workflow_toggles(state)

## Generate `inputs.json`
Build a WDL parameter file `inputs.json` for `TARGET_WORKFLOW`, using the data from your saved session.
You will be prompted to select the sample-subset you would like to use; the JSON will be regenerated 
with the relevant s3 paths. 

Re-run this cell any time you want `inputs.json` for a different subset, or to refresh it after
other changes. File-path inputs (and `job_id`) are always refreshed, since those are recomputed
from the subset/session every time. **NOTE:** Toggles/parameters set in *Workflow Run Options* above
are only refreshed upon request -- and any other non-conflicting edits to the `inputs.json` file 
will be left as-is. Feel free to edit the inputs.json file manually; you can always rerun 
`wb_update_inputs_json_for_subset()` to update the filepaths, without clobbering other settings.
A `.bak` backup is also written before every update.

To target a different workflow, see what's available and update `TARGET_WORKFLOW` in the
*Configuration* section above:

In [ ]:
# wb_list_github_workflows()

In [ ]:
inputs_path <- wb_update_inputs_json_for_subset(state)
inputs_path

## Done
The following `inputs.json` file can be used to run the PANOPLY workflow selected above. Feel free to edit it with additional parameters before runtime; if the JSON needs to be regenerated for a new subset, only filepaths and parameters set within the notebook will be overwritten.

In [ ]:
cat("inputs.json\n  local:", inputs_path, "\n  s3:   ", wb_local_to_s3(inputs_path), "\n")